# Crop ExpW faces for SMIRK

Requires the Kaggle dataset **Expression in-the-Wild (ExpW) Dataset** added as input (*Add Input*).
Run all cells in order. Output: `/kaggle/working/expw_crops.tar` (all crops packed into one file, because Kaggle
does not keep tens of thousands of separate output files) and `/kaggle/working/expw_meta.csv`.

## 1. Locate the input files and check disk space

In [ ]:
import glob, os

parts = sorted(glob.glob('/kaggle/input/**/origin.7z.0*', recursive=True))
labels = glob.glob('/kaggle/input/**/label.lst', recursive=True)
print('archive parts:', *parts, sep='\n  ')
print('label file:', labels)
assert len(parts) == 8, 'expected 8 archive parts - is the ExpW dataset added as input?'
assert len(labels) == 1, 'label.lst not found'
LABEL_FILE = labels[0]

!df -h /tmp /kaggle/working

## 2. Join and extract the 7z parts into /tmp

The first part is named `origin.7z.001.001` on Kaggle, so it is linked as `origin.7z.001` for 7z to find the volumes.
/tmp is not saved after the session; only the crops in /kaggle/working are kept.

In [ ]:
import re, shutil, subprocess

os.makedirs('/tmp/expw_parts', exist_ok=True)
for p in parts:
    number = re.search(r'origin\.7z\.(\d{3})', os.path.basename(p)).group(1)
    link = f'/tmp/expw_parts/origin.7z.{number}'
    if not os.path.exists(link):
        os.symlink(p, link)
print(sorted(os.listdir('/tmp/expw_parts')))

if shutil.which('7z') is None:
    !apt-get install -y -qq p7zip-full

!cd /tmp/expw_parts && 7z x origin.7z.001 -o/tmp/expw -y -bso0 -bsp1

In [ ]:
# find the folder that holds the images listed in label.lst
first_image = open(LABEL_FILE).readline().split()[0]
matches = glob.glob(f'/tmp/expw/**/{first_image}', recursive=True)
assert matches, f'{first_image} not found after extraction'
IMAGE_DIR = os.path.dirname(matches[0])
print('IMAGE_DIR =', IMAGE_DIR, '|', len(os.listdir(IMAGE_DIR)), 'files')

## 3. Write the crop script

In [ ]:
%%writefile /kaggle/working/crop_expw.py
"""
Crop every labeled face of ExpW (Expression in-the-Wild, http://mmlab.ie.cuhk.edu.hk/projects/socialrelation/index.html)
into its own image.

ExpW images are full scenes that may contain several people. label.lst has one line per face:
    image_name face_id_in_image face_box_top face_box_left face_box_right face_box_bottom face_box_cofidence expression_label
with expression labels 0 angry, 1 disgust, 2 fear, 3 happy, 4 sad, 5 surprise, 6 neutral.

Each face is cropped as a square around its box (enlarged by --scale, so that the later MediaPipe-based crop of
BaseDataset still fits), resized to --image_size and saved together with a meta.csv. The crops can then be processed
with apply_mediapipe_to_dataset.py / apply_fan_to_dataset.py.

Output:
    out_dir/images/{idx:06d}.png   cropped faces
    out_dir/meta.csv               name, source image, face id, original box, box confidence, expression label
"""
import os
import csv
import argparse
import numpy as np
import cv2
from tqdm import tqdm

EXPRESSIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']


def parse_line(line):
    tokens = line.strip().split()
    assert len(tokens) == 8, f'unexpected label format ({len(tokens)} tokens): {line}'

    image_name = tokens[0]
    face_id = int(tokens[1])
    top, left, right, bottom = [float(x) for x in tokens[2:6]]
    confidence = float(tokens[6])
    expression = int(tokens[7])
    return image_name, face_id, (left, top, right, bottom), confidence, expression


def crop_face(image, box, scale, image_size):
    # square crop centered on the face box; out-of-image regions are filled with black
    left, top, right, bottom = box
    center = np.array([(left + right) / 2.0, (top + bottom) / 2.0])
    size = max(right - left, bottom - top) * scale

    s = image_size / size
    M = np.array([[s, 0, image_size / 2.0 - s * center[0]],
                  [0, s, image_size / 2.0 - s * center[1]]], dtype=np.float32)

    return cv2.warpAffine(image, M, (image_size, image_size), flags=cv2.INTER_LINEAR, borderValue=0)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Crop the labeled faces of ExpW.')
    parser.add_argument('--image_dir', type=str, required=True, help='Directory with the extracted ExpW images (origin/)')
    parser.add_argument('--label_file', type=str, required=True, help='Path to label.lst')
    parser.add_argument('--out_dir', type=str, required=True, help='Output directory')
    parser.add_argument('--image_size', type=int, default=256, help='Size of the square output crops')
    parser.add_argument('--scale', type=float, default=2.0, help='Crop size relative to the face box size')
    parser.add_argument('--min_face_size', type=float, default=160,
                        help='Skip faces whose box (in original pixels) is smaller than this; small faces are too blurry for SMIRK')
    parser.add_argument('--min_confidence', type=float, default=None, help='Optionally skip faces with a lower box confidence')
    args = parser.parse_args()

    with open(args.label_file) as f:
        lines = [line for line in f if line.strip()]

    os.makedirs(os.path.join(args.out_dir, 'images'), exist_ok=True)

    rows = []
    skipped = {'too_small': 0, 'low_confidence': 0, 'unreadable': 0}
    cached_name, cached_image = None, None  # consecutive lines usually refer to the same image

    for idx, line in enumerate(tqdm(lines)):
        image_name, face_id, box, confidence, expression = parse_line(line)
        left, top, right, bottom = box

        if max(right - left, bottom - top) < args.min_face_size:
            skipped['too_small'] += 1
            continue

        if args.min_confidence is not None and confidence < args.min_confidence:
            skipped['low_confidence'] += 1
            continue

        if image_name != cached_name:
            cached_name, cached_image = image_name, cv2.imread(os.path.join(args.image_dir, image_name))
        if cached_image is None:
            skipped['unreadable'] += 1
            continue

        cropped = crop_face(cached_image, box, args.scale, args.image_size)

        name = f'{idx:06d}'
        cv2.imwrite(os.path.join(args.out_dir, 'images', name + '.png'), cropped)
        rows.append([name, image_name, face_id, left, top, right, bottom, confidence, expression, EXPRESSIONS[expression]])

    with open(os.path.join(args.out_dir, 'meta.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['name', 'source_image', 'face_id', 'left', 'top', 'right', 'bottom', 'confidence', 'label', 'expression'])
        writer.writerows(rows)

    print(f'Saved {len(rows)} faces, skipped {skipped}')
    counts = np.bincount([r[8] for r in rows], minlength=len(EXPRESSIONS))
    print('Faces per expression:', dict(zip(EXPRESSIONS, counts.tolist())))

## 4. Crop the faces

Faces smaller than 160 px are skipped (too blurry for SMIRK). Change `--min_face_size` if needed.

In [ ]:
!python /kaggle/working/crop_expw.py --image_dir "$IMAGE_DIR" --label_file "$LABEL_FILE" --out_dir /tmp/expw_crops --min_face_size 160

## 5. Look at some crops

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, cv2

meta = pd.read_csv('/tmp/expw_crops/meta.csv', dtype={'name': str})
print(meta['expression'].value_counts())

sample = meta.sample(min(24, len(meta)), random_state=0)
fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img = cv2.imread(f"/tmp/expw_crops/images/{row['name']}.png")[..., ::-1]
    ax.imshow(img); ax.set_title(row['expression']); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Pack the crops into a single file

In [ ]:
!cp /tmp/expw_crops/meta.csv /kaggle/working/expw_meta.csv
!tar -cf /kaggle/working/expw_crops.tar -C /tmp expw_crops
!ls -lh /kaggle/working

## 7. Save the result

*Save Version → Save & Run All*. When it finishes, open the notebook's **Output** tab and choose **New Dataset**
so later notebooks can use the crops without extracting the 8 GB archive again
(`!tar -xf /kaggle/input/<dataset>/expw_crops.tar -C /tmp` takes about a minute).